# Reference

https://dacon.io/competitions/official/235614/codeshare/1300


# Download Datasets

In [1]:
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

# Normalize data with mean=0.5, std=1.0
mnist_transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize((0.5,), (1.0,))
])

In [2]:
download_root = '../datasets/MNIST_DATASET'

In [3]:
train_dataset = MNIST(download_root, transform=mnist_transform, train=True, download=True)
valid_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)
test_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)

In [4]:
from torch.utils.data import DataLoader

import os
import random
import torch
import warnings

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')

# Set Seed

In [5]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

# Utils

In [6]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

# Models 

In [7]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [8]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

# Train/Eval Functions

In [9]:
def train_loop_classifier(feature_extractor, classifier, train_loader, loss_func, optimizer, 
                          summary_loss, summary_acc=None, device=None):
    feature_extractor.train()
    classifier.train()
    
    for batch_idx, (data, target) in enumerate(train_loader):
        if device is not None:
            data, target = data.to(device), target.to(device)
            
        optimizer.zero_grad()
        
        feature = feature_extractor(data)
        feature = torch.flatten(feature, 1)
        output = classifier(feature)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        
        summary_loss.update(loss.detach().item(), BATCH_SIZE)
        if summary_acc is not None:
            summary_acc.update(acc, BATCH_SIZE)
        
    return summary_loss, summary_acc

def eval_loop_classifier(feature_extractor, classifier, valid_loader, loss_func, optimizer, 
                         summary_loss, summary_acc=None, device=None):
    feature_extractor.eval()
    classifier.eval()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            if device is not None:
                data, target = data.to(device), target.to(device)

            feature = feature_extractor(data)
            feature = torch.flatten(feature, 1)
            output = classifier(feature)

            loss = loss_func(output, target)

            summary_loss.update(loss.detach().item(), BATCH_SIZE)
            if summary_acc is not None:
                summary_acc.update(acc, BATCH_SIZE)

    return summary_loss, summary_acc

# Training with Early Stopping

In [10]:
BATCH_SIZE = 64
LR = 0.001
ES_THRES = 10
SEED = 42
seed_everything(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = FeatureExtractor(in_channel_dim=1)
classifier = Classifier(256)

feature_extractor.to(device)
classifier.to(device)

criterion = nn.CrossEntropyLoss()
model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
optimizer = optim.Adam(model_parameters, lr=LR)


train_loader = DataLoader(dataset=train_dataset, 
                         batch_size=BATCH_SIZE,
                         shuffle=True)

valid_loader = DataLoader(dataset=test_dataset, 
                         batch_size=BATCH_SIZE,
                         shuffle=True)

test_loader = DataLoader(dataset=test_dataset, 
                         batch_size=BATCH_SIZE,
                         shuffle=True)

best_epoch = None
best_loss = None
epoch = 0
es_count = 0
while(True):
    epoch += 1
    summary_loss_train = AverageMeter()
#     summary_acc_train = AverageMeter()
    summary_loss_valid = AverageMeter()
#     summary_acc_valid = AverageMeter()
    
    summary_loss_train, _ = train_loop_classifier(feature_extractor, classifier, train_loader, 
                                               criterion, optimizer, summary_loss_train, None, device=device)
    summary_loss_valid, _ = eval_loop_classifier(feature_extractor, classifier, valid_loader, 
                                               criterion, optimizer, summary_loss_valid, None, device=device)
    
    print(f"[epoch]{epoch} [train]{summary_loss_train.avg} [valid]{summary_loss_valid.avg}")
    
    if best_loss is None:
        best_loss = summary_loss_valid.avg
        best_epoch = epoch
    
    if best_loss > summary_loss_valid.avg:
        best_loss = summary_loss_valid.avg
        best_epoch = epoch
        es_count = 0
    else:
        es_count += 1
    
    if es_count == ES_THRES:
        break

print(f"Best epoch: {best_epoch}")